# 03c — RM-c: Frozen Encoder + Retrieval-Augmented Classification (RAC)

Skenario terakhir. **Tanpa training IndoBERT baru** — memakai ulang dari RM-b:
- embedding beku (mean-pool) `results/features/{train,val,test}_emb.npy`
- head terlatih `results/checkpoints/rmb/head_best.pt`

**Alur RAC per sampel:** `p_bert = softmax(head(emb))` · cari k tetangga di **FAISS (index dari TRAIN saja)** → distribusi label tetangga `p_retr` · fusi `p_final = (1-α)·p_bert + α·p_retr`.

> **Anti-leakage:** index FAISS **hanya** dari train (near-duplicate train↔test sudah dibuang di preprocessing NFKC-dedup + guard). Tanpa ini, retrieval jadi "curang".

Notebook ini berjalan **murni dari cache** (tak perlu memuat IndoBERT) → cepat. Butuh `faiss-cpu`.

## 1. Setup Colab

In [1]:
import os, sys, json, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Bukan Colab — pakai path lokal.')

if IN_COLAB:
    os.system('pip install -q faiss-cpu transformers scikit-learn')

PROJECT_DIR = Path('/content/drive/MyDrive/IndoBERT-with-RAC') if IN_COLAB else Path('..')
SRC_DIR = PROJECT_DIR / 'src'
assert SRC_DIR.exists(), f'src tidak ditemukan di {SRC_DIR} — cek PROJECT_DIR'
sys.path.insert(0, str(SRC_DIR))

import numpy as np
import torch
np.random.seed(42); torch.manual_seed(42)
print('siap.')

Bukan Colab — pakai path lokal.


siap.


## 2. Konfigurasi (grid tuning α & k)

In [2]:
CFG = dict(
    scenario   = 'RM-c',
    weighting  = 'similarity',   # bobot tetangga = cosine (max(sim,0)); 'uniform' utk voting rata
    k_default  = 5,              # default CLAUDE.md
    alpha_default = 0.3,         # default CLAUDE.md
    k_grid     = [1, 3, 5, 10, 20, 50],
    alpha_grid = [round(a,2) for a in np.arange(0.0, 1.01, 0.1)],
)
FEAT_DIR = PROJECT_DIR / 'results' / 'features'
CKPT_RMB = PROJECT_DIR / 'results' / 'checkpoints' / 'rmb' / 'head_best.pt'
MET_DIR  = PROJECT_DIR / 'results' / 'metrics'
FIG_DIR  = PROJECT_DIR / 'results' / 'figures'
for d in (MET_DIR, FIG_DIR): d.mkdir(parents=True, exist_ok=True)
print(json.dumps({k:v for k,v in CFG.items() if k!='alpha_grid'}, indent=2))

{
  "scenario": "RM-c",
  "weighting": "similarity",
  "k_default": 5,
  "alpha_default": 0.3,
  "k_grid": [
    1,
    3,
    5,
    10,
    20,
    50
  ]
}


## 3. Muat Cache (embedding + head RM-b) → hitung p_bert

In [3]:
import rac
import evaluate as E
from modeling import FrozenHead

emb = {s: np.load(FEAT_DIR / f'{s}_emb.npy') for s in ['train','val','test']}
lab = {s: np.load(FEAT_DIR / f'{s}_label.npy') for s in ['train','val','test']}
for s in ['train','val','test']:
    print(f'{s}: emb {emb[s].shape} | label dist {np.bincount(lab[s]).tolist()}')

ck = torch.load(CKPT_RMB, map_location='cpu')
head = FrozenHead(hidden_size=ck['hidden_size']); head.load_state_dict(ck['head_state']); head.eval()

def p_bert(split):
    with torch.no_grad():
        return rac.softmax(head(torch.tensor(emb[split])).numpy())
pb = {s: p_bert(s) for s in ['val','test']}
print('p_bert siap (val/test).')

C:\Penelitian\IndoBERT-with-RAC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train: emb (6588, 768) | label dist [5391, 1197]
val: emb (1402, 768) | label dist [1145, 257]
test: emb (1405, 768) | label dist [1149, 256]
p_bert siap (val/test).


## 4. Bangun Index FAISS (dari TRAIN saja)

In [4]:
t0 = time.perf_counter()
index, _ = rac.build_faiss_index(emb['train'])
build_time = time.perf_counter() - t0
index_mb = emb['train'].nbytes / (1024**2)
print(f'index size: {index.ntotal} vektor | build {build_time*1000:.1f} ms | ~{index_mb:.1f} MB')

index size: 6588 vektor | build 37.3 ms | ~19.3 MB


## 5. Tuning (α, k) pada VALIDATION

Dipilih kombinasi dengan **val F1-macro** tertinggi (test tidak disentuh saat tuning).

In [5]:
grid = {}
for k in CFG['k_grid']:
    sims, idx = rac.retrieve(index, emb['val'], k)
    neigh = lab['train'][idx]
    p_retr = rac.retrieval_distribution(sims, neigh, num_labels=2, weighting=CFG['weighting'])
    for a in CFG['alpha_grid']:
        pred = rac.fuse(pb['val'], p_retr, a).argmax(1)
        grid[(k,a)] = E.classification_metrics(lab['val'], pred)['f1_macro']

best_f1, best_k, best_a = max((v,k,a) for (k,a),v in grid.items())
print(f'BEST on VAL: k={best_k}, alpha={best_a} -> val F1-macro {best_f1:.4f}')
print(f"Referensi default (k=5, alpha=0.3): val F1-macro {grid[(5,0.3)]:.4f}")

# heatmap tuning
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
M = np.array([[grid[(k,a)] for a in CFG['alpha_grid']] for k in CFG['k_grid']])
fig, ax = plt.subplots(figsize=(9,3.5))
im = ax.imshow(M, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(CFG['alpha_grid'])), [f'{a:.1f}' for a in CFG['alpha_grid']])
ax.set_yticks(range(len(CFG['k_grid'])), CFG['k_grid'])
ax.set_xlabel('alpha'); ax.set_ylabel('k'); ax.set_title('RM-c tuning — val F1-macro')
bi = CFG['k_grid'].index(best_k); bj = CFG['alpha_grid'].index(best_a)
ax.scatter([bj],[bi], marker='*', s=200, c='red', edgecolor='white')
fig.colorbar(im, fraction=0.046, pad=0.04); fig.tight_layout()
fig.savefig(FIG_DIR / 'rmc_tuning.png', dpi=120, bbox_inches='tight'); plt.close(fig)
print('grid (k=best) vs alpha:', {a: round(grid[(best_k,a)],4) for a in CFG['alpha_grid']})

BEST on VAL: k=3, alpha=0.5 -> val F1-macro 0.9577
Referensi default (k=5, alpha=0.3): val F1-macro 0.9393


grid (k=best) vs alpha: {np.float64(0.0): 0.9063, np.float64(0.1): 0.9172, np.float64(0.2): 0.9356, np.float64(0.3): 0.9409, np.float64(0.4): 0.9515, np.float64(0.5): 0.9577, np.float64(0.6): 0.9525, np.float64(0.7): 0.9448, np.float64(0.8): 0.9397, np.float64(0.9): 0.9397, np.float64(1.0): 0.9397}


## 6. Evaluasi TEST dengan (α, k) terbaik

In [6]:
preds, p_final = rac.rac_predict(index, lab['train'], emb['test'], pb['test'],
                                 k=best_k, alpha=best_a, weighting=CFG['weighting'])
metrics = E.classification_metrics(lab['test'], preds)
print(f'=== RM-c TEST (k={best_k}, alpha={best_a}) ===')
for key in ['accuracy','f1_macro','precision_macro','recall_macro','f1_class1','precision_class1','recall_class1']:
    print(f'  {key:18s}: {metrics[key]:.4f}')
E.plot_confusion_matrix(lab['test'], preds, FIG_DIR / 'rmc_confusion.png',
                        title=f'RM-c (RAC) — Confusion Matrix (k={best_k}, a={best_a})')

# perbandingan: RM-b (head, alpha=0) & RM-c default
base = E.classification_metrics(lab['test'], pb['test'].argmax(1))
pr_def,_ = rac.rac_predict(index, lab['train'], emb['test'], pb['test'], k=CFG['k_default'], alpha=CFG['alpha_default'])
mdef = E.classification_metrics(lab['test'], pr_def)
print(f"\nRM-b (head, alpha=0)          : F1-macro {base['f1_macro']:.4f}")
print(f"RM-c default (k=5, alpha=0.3) : F1-macro {mdef['f1_macro']:.4f}")
print(f"RM-c tuned  (k={best_k}, alpha={best_a}) : F1-macro {metrics['f1_macro']:.4f}")

=== RM-c TEST (k=3, alpha=0.5) ===
  accuracy          : 0.9701
  f1_macro          : 0.9500
  precision_macro   : 0.9486
  recall_macro      : 0.9514
  f1_class1         : 0.9183
  precision_class1  : 0.9147
  recall_class1     : 0.9219



RM-b (head, alpha=0)          : F1-macro 0.9006
RM-c default (k=5, alpha=0.3) : F1-macro 0.9304
RM-c tuned  (k=3, alpha=0.5) : F1-macro 0.9500


## 7. Efisiensi + Perbandingan RM-a / RM-b / RM-c

In [7]:
import pandas as pd

# latency overhead RAC (retrieval + fusi) pada 1 query — di ATAS biaya encoder (sama seperti RM-b)
q = emb['test'][:1]
def _rac_step(_):
    s,i = rac.retrieve(index, q, best_k)
    pr = rac.retrieval_distribution(s, lab['train'][i], 2, CFG['weighting'])
    return rac.fuse(pb['test'][:1], pr, best_a)
rac_overhead_ms = E.measure_latency(_rac_step, None, n_warmup=5, n_runs=200)

row = {'scenario':'RM-c', **{k: round(v,6) for k,v in metrics.items()},
       'best_k': best_k, 'best_alpha': best_a, 'weighting': CFG['weighting'],
       'trainable_params_new': 0, 'reused_head_params': int(ck['hidden_size']*2+2),
       'index_vectors': int(index.ntotal), 'index_build_ms': round(build_time*1000,2),
       'rac_overhead_ms_per_sample': round(rac_overhead_ms,4),
       'val_f1_macro_best': round(best_f1,6),
       'f1_macro_default_5_0.3': round(mdef['f1_macro'],6)}
E.save_metrics_csv(row, MET_DIR / 'rmc_metrics.csv')

# tabel perbandingan (baca metrik RM-a/RM-b bila ada)
def load1(p):
    return pd.read_csv(p).iloc[0].to_dict() if Path(p).exists() else {}
ra = load1(MET_DIR/'rma_metrics.csv'); rb = load1(MET_DIR/'rmb_metrics.csv')
comp = pd.DataFrame([
    {'model':'RM-a (full FT)', 'F1_macro': ra.get('f1_macro'), 'F1_judi': ra.get('f1_class1'),
     'prec_judi': ra.get('precision_class1'), 'trainable_params': ra.get('trainable_params'),
     'train_time_s': ra.get('total_train_time_s')},
    {'model':'RM-b (frozen+head)', 'F1_macro': rb.get('f1_macro'), 'F1_judi': rb.get('f1_class1'),
     'prec_judi': rb.get('precision_class1'), 'trainable_params': rb.get('trainable_params'),
     'train_time_s': rb.get('total_train_time_s')},
    {'model':f'RM-c (RAC k={best_k},a={best_a})', 'F1_macro': round(metrics['f1_macro'],4),
     'F1_judi': round(metrics['f1_class1'],4), 'prec_judi': round(metrics['precision_class1'],4),
     'trainable_params': 0, 'train_time_s': 0},
])
print(comp.to_string(index=False))

# kriteria sukses RM-c vs RM-a
if ra:
    gap = ra['f1_macro'] - metrics['f1_macro']
    print(f"\nKriteria sukses RM-c vs RM-a:")
    print(f"  F1-macro gap        : {gap*100:.2f} pp  ({'LOLOS' if gap<=0.03 else 'GAGAL'} <=3pp)")
    print(f"  Reduksi trainable   : 100% (0 param baru dilatih)  (LOLOS >=90%)")
    print(f"  Reduksi waktu latih : 100% (tanpa training)         (LOLOS >=50%)")
print('\nTersimpan: rmc_metrics.csv, rmc_confusion.png, rmc_tuning.png')

               model  F1_macro  F1_judi  prec_judi  trainable_params  train_time_s
      RM-a (full FT)  0.965104 0.942801   0.952191         109485314         285.8
  RM-b (frozen+head)  0.900625 0.840787   0.775578              1538          23.9
RM-c (RAC k=3,a=0.5)  0.950000 0.918300   0.914700                 0           0.0

Kriteria sukses RM-c vs RM-a:
  F1-macro gap        : 1.51 pp  (LOLOS <=3pp)
  Reduksi trainable   : 100% (0 param baru dilatih)  (LOLOS >=90%)
  Reduksi waktu latih : 100% (tanpa training)         (LOLOS >=50%)

Tersimpan: rmc_metrics.csv, rmc_confusion.png, rmc_tuning.png


## Ringkasan

RM-c (RAC) menutup sebagian besar celah RM-b **tanpa training IndoBERT tambahan** — hanya retrieval k-NN atas embedding train + fusi probabilitas.

- Kelemahan RM-b = **precision kelas judi** (banyak false positive). Retrieval menilai tetangga nyata → memangkas FP.
- Hyperparameter final **α & k** ditentukan dari validation → catat ke `DATASET.md`.
- **Anti-leakage**: index FAISS hanya dari train; near-duplicate train↔test sudah dibuang saat preprocessing.

**Catatan efisiensi:** RM-c menambah 0 parameter terlatih & 0 waktu training (pakai ulang RM-b). Overhead hanya pencarian FAISS + fusi (sub-milidetik/sampel) di atas biaya encoder yang sama dengan RM-b. Untuk angka latency/memori final Bab 4, ukur ketiga model dalam satu sesi GPU yang sama.